In [1]:
"""

import pandas as pd
from pathlib import Path
import sys

def processa_per_sessione_nonna(base_repo_path, output_base_dir):
    """
    Esegue i seguenti passaggi:
    1. Itera su ogni cartella Soggetto (es. 'S01', 'S02').
    2. All'interno di ogni Soggetto, itera su ogni sottocartella "sessione"
       (la "nonna", es. 'test1a', 'test1b').
    3. Per CIASCUNA sessione:
       a. Trova tutti i file 'predictions_log.csv' al suo interno
          (a qualsiasi livello di profondità).
       b. Estrae le colonne 'label' e 'timestamp'.
       c. Concatena tutti i dati di quella sessione.
       d. Ordina il DataFrame combinato in base a 'timestamp'.
       e. Salva il file in '.../data/Ssoggetto/steady_state/'
          con il nome 'predictions_log_nomesessione.csv'.
    """
    
    # 1. Definizione dei percorsi
    
    # Percorso INPUT: Dove si trovano i dati
    base_path = Path(base_repo_path)
    
    # Percorso OUTPUT: Percorso base assoluto specificato dall'utente
    output_base_path = Path(output_base_dir)
    
    # Colonne da estrarre
    colonne_richieste = ['class', 'timestamp']

    print(f"Inizio ricerca dati in: {base_path}")
    print(f"Percorso base di output: {output_base_path}\n")

    # 2. Trova tutte le cartelle Soggetto
    s_folders = sorted([f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')])
    
    if not s_folders:
        print("Attenzione: Nessuna cartella che inizia con 'S' trovata.")
        return

    print(f"Trovate {len(s_folders)} cartelle soggetto. Inizio elaborazione...")

    # 3. Itera su ciascun Soggetto (es. S03)
    for subject_folder in s_folders:
        subject_name = subject_folder.name
        print(f"\n--- Processando Soggetto: {subject_name} ---")
        
        # Definisci la cartella di output base per questo soggetto
        subject_output_dir = output_base_path / subject_name / "steady_state"
        
        # Crea la cartella di output se non esiste
        try:
            subject_output_dir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            print(f"  > ERRORE: Impossibile creare la cartella di output {subject_output_dir}")
            print(f"    Dettagli: {e}")
            continue # Salta al prossimo soggetto
            
        # 4. Trova tutte le cartelle "sessione" (nonne) dentro il soggetto
        session_folders = [f for f in subject_folder.iterdir() if f.is_dir()]
        
        if not session_folders:
            print(f"  > Attenzione: Nessuna cartella 'sessione' trovata per {subject_name}.")
            continue
            
        print(f"  > Trovate {len(session_folders)} sessioni: {[f.name for f in session_folders]}")

        # 5. Itera su ciascuna Sessione (es. test1a)
        for session_folder in session_folders:
            session_name = session_folder.name
            print(f"    > Processando sessione: {session_name}")
            
            # Lista per i dati di QUESTA sessione
            session_data_list = []
            
            # Nome file di output basato sulla sessione
            output_csv_file = subject_output_dir / f"predictions_log_{session_name}.csv"
            
            # 6. Cerca tutti i CSV all'interno della cartella sessione (ricorsivamente)
            csv_files_in_session = list(session_folder.rglob("predictions_log.csv"))
            
            if not csv_files_in_session:
                print(f"      > Nessun 'predictions_log.csv' trovato in {session_name}.")
                continue # Passa alla prossima sessione

            file_trovati = 0
            for csv_file in csv_files_in_session:
                try:
                    df = pd.read_csv(
                        csv_file, 
                        usecols=colonne_richieste, 
                        skipinitialspace=True,
                        dtype={'timestamp': str}
                    )
                    
                    if not df.empty:
                        session_data_list.append(df)
                        file_trovati += 1
                    
                except pd.errors.EmptyDataError:
                    print(f"      > Attenzione: File vuoto ignorato {csv_file.relative_to(session_folder)}")
                except ValueError as e:
                    print(f"      > ERRORE: Colonne non trovate in {csv_file.relative_to(session_folder)}.")
                except Exception as e:
                    print(f"      > Errore durante lettura di {csv_file.relative_to(session_folder)}: {e}")
            
            if not session_data_list:
                print(f"      > Nessun dato valido letto per la sessione {session_name}.")
                continue
                
            print(f"      > Letti {file_trovati} file. Ora concateno e ordino...")

            # 7. Concatena, Ordina e Salva per QUESTA sessione
            try:
                combined_df = pd.concat(session_data_list, ignore_index=True)
                combined_df_sorted = combined_df.sort_values(by='timestamp')
                
                # Salva CSV
                combined_df_sorted.to_csv(output_csv_file, index=False)
                print(f"      > File salvato con successo per {session_name} in:\n        {output_csv_file}")
                
            except Exception as e:
                print(f"      > ERRORE durante il salvataggio del file per {session_name}: {e}")

    print("\n--- Elaborazione per sessione completata ---")

# --- INIZIO SCRIPT ---
if __name__ == "__main__":
    
    # Percorso INPUT: dove si trovano i dati da analizzare.
    percorso_base_dati = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    # Percorso OUTPUT: la cartella base (assoluta) dove salvare i risultati
    percorso_base_output = r"C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data"
    
    processa_per_sessione_nonna(percorso_base_dati, percorso_base_output)"""

Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Percorso base di output: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data

Trovate 12 cartelle soggetto. Inizio elaborazione...

--- Processando Soggetto: S03 ---
  > Trovate 5 sessioni: ['calib', 'test1a', 'test1b', 'test2a', 'test2b']
    > Processando sessione: calib
      > Nessun 'predictions_log.csv' trovato in calib.
    > Processando sessione: test1a
      > Letti 1 file. Ora concateno e ordino...
      > File salvato con successo per test1a in:
        C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\predictions_log_test1a.csv
    > Processando sessione: test1b
      > Letti 1 file. Ora concateno e ordino...
      > File salvato con successo per test1b in:
        C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\predictions_log_test1b.csv
    > Processando sessione: test2a
      > Letti 1 file. Ora concateno e ordino...
      > File salvato con successo per test2a in:
  

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import (
    # accuracy_score, # Rimosso
    # classification_report, # Rimosso
    confusion_matrix
)
# import sys # Rimosso, non utilizzato

def plot_confusion_matrix(cm_abs, class_names, title, output_path):
    """
    Funzione helper per creare e salvare un plot della matrice di confusione.
    Mostra le percentuali (normalizzate per riga) con font grandi.
    """
    try:
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Calcola le percentuali normalizzate per riga (per 'True Label')
        cm_sum = cm_abs.sum(axis=1)[:, np.newaxis]
        
        # Evita divisione per zero se una classe non ha campioni reali
        cm_perc = np.nan_to_num(cm_abs.astype('float') / cm_sum)
        
        # Formattazione per le annotazioni (es. "95.20%")
        plot_fmt = '.2%' 
        
        # Set di font
        annot_font_size = 22
        title_font_size = 26
        label_font_size = 24
        tick_font_size = 20
        
        sns.heatmap(
            cm_perc, 
            annot=True, 
            fmt=plot_fmt, 
            cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names,
            ax=ax,
            annot_kws={"size": annot_font_size} # Imposta font celle
        )
        
        ax.set_xlabel('Etichetta Predetta (Predicted Label)', fontsize=label_font_size)
        ax.set_ylabel('Etichetta Reale (True Label)', fontsize=label_font_size)
        ax.set_title(title, fontsize=title_font_size)
        
        ax.tick_params(axis='x', labelsize=tick_font_size)
        ax.tick_params(axis='y', labelsize=tick_font_size, rotation=0)
        
        plt.tight_layout()
        plt.savefig(output_path)
        plt.close(fig) 
        
        print(f"  > Plot matrice di confusione (PERCENTUALE) salvato in:\n    {output_path}")

    except Exception as e:
        print(f"  > ERRORE durante la creazione del plot: {e}")

def valuta_prestazioni_soggetto(soggetto_id, base_data_dir):
    """
    Genera un report di valutazione per un singolo soggetto.
    Cerca i file 'predictions_log_*.csv', calcola le metriche
    e salva i risultati in cartelle dedicate.
    """
    
    # 1. Definizione dei percorsi
    input_dir = Path(base_data_dir) / soggetto_id / "steady_state"
    output_eval_base_dir = input_dir / "evaluation"
    
    CLASS_NAMES = ['no weight', 'light', 'medium', 'heavy']
    CLASS_LABELS = [0, 1, 2, 3]

    print(f"--- Inizio Valutazione per Soggetto: {soggetto_id} ---")
    print(f"Cartella di input: {input_dir}")
    print(f"Cartella base di output: {output_eval_base_dir}\n")

    # 2. Trova tutti i file log delle sessioni
    log_files = list(input_dir.glob("predictions_log_*.csv"))
    
    if not log_files:
        print(f"Attenzione: Nessun file 'predictions_log_*.csv' trovato in {input_dir}")
        return

    print(f"Trovati {len(log_files)} file di log da analizzare...")

    # 3. Itera su ciascun file di log
    for file_path in log_files:
        
        session_name = file_path.stem.replace("predictions_log_", "")
        print(f"\n--- Processando sessione: {session_name} ---")

        # 4. Crea la cartella di output dedicata per questa sessione
        session_output_dir = output_eval_base_dir / session_name
        try:
            session_output_dir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            print(f"  > ERRORE: Impossibile creare cartella di output {session_output_dir}")
            print(f"    Dettagli: {e}")
            continue 

        # 5. Leggi i dati
        try:
            df = pd.read_csv(file_path)
            
            if 'class' not in df.columns or 'class_real' not in df.columns:
                print(f"  > ERRORE: Il file {file_path.name} non contiene 'class' o 'class_real'. Salto.")
                continue
                
            df.dropna(subset=['class_real'], inplace=True)
            
            y_true = df['class_real'].astype(int)
            y_pred = df['class'].astype(int)
            
        except Exception as e:
            print(f"  > ERRORE durante la lettura del file {file_path.name}: {e}")
            continue

        if y_true.empty:
            print(f"  > Attenzione: Nessun dato valido (senza NaN) trovato in {file_path.name}.")
            continue
            
        # 6. Calcola Metriche (Solo Matrice di Confusione)
        print(f"  > Calcolo matrice di confusione per {len(y_true)} campioni...")
        
        # --- BLOCCO METRICHE RIMOSSO ---
        # accuracy = accuracy_score(y_true, y_pred)
        # report_str = classification_report(...)
        # --- FINE BLOCCO RIMOSSO ---
        
        # Matrice di Confusione (valori assoluti)
        # Questa la calcoliamo sempre per salvarla nel CSV e per il plot
        cm_abs = confusion_matrix(
            y_true, y_pred, 
            labels=CLASS_LABELS
        )

        # 7. Salva Report Metriche (Task 3)
        # --- BLOCCO SALVATAGGIO REPORT .TXT RIMOSSO ---

        # 8. Salva Matrice di Confusione in CSV (Task 4)
        cm_csv_path = session_output_dir / "confusion_matrix_absolute.csv"
        try:
            cm_df = pd.DataFrame(
                cm_abs, 
                index=[f"True_{name}" for name in CLASS_NAMES], 
                columns=[f"Pred_{name}" for name in CLASS_NAMES]
            )
            cm_df.to_csv(cm_csv_path)
            print(f"  > Matrice di confusione (CSV Assoluta) salvata in: {cm_csv_path.name}")
        except Exception as e:
            print(f"  > ERRORE salvataggio CSV matrice: {e}")

        # 9. Crea e Salva Plot (Task 2)
        plot_png_path = session_output_dir / "confusion_matrix_plot.png"
        
        plot_title = f"Matrice di Confusione - {session_name}\nSoggetto {soggetto_id} (Normalizzata per Riga %)"
        
        plot_confusion_matrix(
            cm_abs, # Passiamo i valori assoluti
            CLASS_NAMES, 
            plot_title, 
            plot_png_path
        )

    # print(f"\n--- Valutazione completata per {soggetto_id} ---") # Modificato nel main loop

# --- INIZIO SCRIPT ---
if __name__ == "__main__":
    
    # Percorso base dove si trovano le cartelle S01, S02, S03...
    PERCORSO_BASE_DATA = r"C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data"
    base_data_path = Path(PERCORSO_BASE_DATA)

    if not base_data_path.is_dir():
        print(f"ERRORE: Il percorso base non esiste: {PERCORSO_BASE_DATA}")
    else:
        print(f"Avvio elaborazione batch nella cartella: {base_data_path}\n")
        
        # Cerca tutte le cartelle che iniziano con 'S' (es. S01, S02, ...)
        # e assicurati che siano directory. Ordinale.
        soggetti_trovati = sorted(
            [d for d in base_data_path.glob("S*") if d.is_dir()]
        )
        
        if not soggetti_trovati:
            print(f"Nessuna cartella soggetto (S*) trovata in {base_data_path}")
        else:
            print(f"Trovati {len(soggetti_trovati)} soggetti: {[s.name for s in soggetti_trovati]}\n")
            
            # Itera su ogni soggetto e avvia la valutazione
            for soggetto_path in soggetti_trovati:
                soggetto_id = soggetto_path.name
                valuta_prestazioni_soggetto(soggetto_id, PERCORSO_BASE_DATA)
                print(f"--- Fine valutazione per {soggetto_id} ---\n")
        
        print("--- Elaborazione batch completata ---")

Avvio elaborazione batch nella cartella: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data

Trovati 12 soggetti: ['S03', 'S08', 'S10', 'S12', 'S13', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21']

--- Inizio Valutazione per Soggetto: S03 ---
Cartella di input: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state
Cartella base di output: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\evaluation

Trovati 4 file di log da analizzare...

--- Processando sessione: test1a ---
  > Calcolo matrice di confusione per 3196 campioni...
  > Matrice di confusione (CSV Assoluta) salvata in: confusion_matrix_absolute.csv
  > Plot matrice di confusione (PERCENTUALE) salvato in:
    C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\evaluation\test1a\confusion_matrix_plot.png

--- Processando sessione: test1b ---
  > Calcolo matrice di confusione per 2769 campioni...
  > Matrice di confusione (CSV Assoluta) salvata in: confusion_matrix_absolute.csv
  >

In [3]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns  # Importato Seaborn
import numpy as np
import random
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support  # Aggiunto per le metriche
)

# --- GLOBAL CONSTANTS ---
CLASS_NAMES_FULL = ['no weight', 'light', 'medium', 'heavy']
CLASS_LABELS_FULL = [0, 1, 2, 3] 
TEST_SESSIONS = ['test1a', 'test1b', 'test2a', 'test2b']

# Costanti per analisi 3-class (reintrodotte)
CLASS_NAMES_3 = ['light', 'medium', 'heavy']
CLASS_LABELS_3 = [1, 2, 3]

# Costanti per il plotting (come da tua richiesta)
DPI = 300
FIGSIZE = (8, 6) 

# ==================
# --- FUNZIONI HELPER (CSV, METRICHE, UNDERSAMPLING) ---
# ==================

def save_cm_csv_only(cm_abs, output_dir, file_name_prefix, class_names):
    """Saves a confusion matrix (absolute) to CSV only."""
    cm_csv_path = output_dir / f"{file_name_prefix}_absolute.csv"
    try:
        cm_df = pd.DataFrame(
            cm_abs, 
            index=[f"True_{name}" for name in class_names], 
            columns=[f"Pred_{name}" for name in class_names]
        )
        cm_df.to_csv(cm_csv_path)
        print(f"    > CSV saved: {cm_csv_path.name}")
    except Exception as e:
        print(f"    > ERROR saving CSV: {e}")

def reconstruct_predictions(cm):
    """
    Reconstructs y_true and y_pred lists from a confusion matrix.
    Returns shuffled lists.
    """
    pairs = []
    for true_idx, row in enumerate(cm):
        for pred_idx, count in enumerate(row):
            # Add 'count' pairs of (true_idx, pred_idx)
            pairs.extend([(true_idx, pred_idx)] * int(count))
    
    # Shuffle the pairs
    random.shuffle(pairs)
    
    y_true = [p[0] for p in pairs]
    y_pred = [p[1] for p in pairs]
    return y_true, y_pred

def perform_undersampling(y_true, y_pred, class_labels):
    """
    Performs undersampling based on the minority class (y_true).
    """
    data = list(zip(y_true, y_pred))
    
    # Calculate counts for each REAL class
    true_counts = np.bincount(y_true, minlength=len(class_labels))
    
    # Find the minority class size (excluding classes with 0 samples)
    min_samples = np.min(true_counts[true_counts > 0])
    
    print(f"    > Performing undersampling: {min_samples} samples per class.")
    
    undersampled_data = []
    # Dictionary to track how many samples we have for each class
    class_counters = {label: 0 for label in class_labels}
    
    # Iterate over shuffled data
    for yt, yp in data:
        # If we haven't reached the limit for this true class
        if class_counters[yt] < min_samples:
            undersampled_data.append((yt, yp))
            class_counters[yt] += 1
            
    if not undersampled_data:
        return [], []
        
    y_true_us = [p[0] for p in undersampled_data]
    y_pred_us = [p[1] for p in undersampled_data]
    
    return y_true_us, y_pred_us

def calculate_and_format_metrics(y_true_us, y_pred_us, title, class_labels_4, class_names_4, class_labels_3, class_names_3):
    """
    Calculates and formats metrics for 4-class and 3-class analysis.
    """
    if not y_true_us:
        return f"--- {title} ---\nNo data to analyze.\n\n"

    buffer = []
    buffer.append("=" * 80)
    buffer.append(f"--- METRICS FOR: {title} ---")
    buffer.append("=" * 80)
    buffer.append(f"(Based on {len(y_true_us)} total samples after undersampling)\n")

    # --- 1. 4-Class Metrics ---
    buffer.append("\n--- 4-Class Analysis (no weight, light, medium, heavy) ---")
    
    acc_4_class = accuracy_score(y_true_us, y_pred_us)
    report_4_class = classification_report(
        y_true_us, y_pred_us,
        labels=class_labels_4,
        target_names=class_names_4,
        zero_division=0
    )
    buffer.append(f"\nOverall Accuracy (4-Class): {acc_4_class:.4f}")
    buffer.append("\nClassification Report (4-Class):\n")
    buffer.append(report_4_class)

    # --- 2. 3-Class Metrics (light, medium, heavy) ---
    buffer.append("\n\n--- 3-Class Analysis (light, medium, heavy) ---")
    
    # Filter the lists to calculate 3-class accuracy
    y_true_3class = []
    y_pred_3class = []
    for yt, yp in zip(y_true_us, y_pred_us):
        # Consider only samples that *should* be 1, 2, or 3
        if yt in class_labels_3:
            y_true_3class.append(yt)
            y_pred_3class.append(yp)

    if y_true_3class:
        acc_3_class = accuracy_score(y_true_3class, y_pred_3class)
        buffer.append(f"\nOverall Accuracy (Classes 1, 2, 3 only): {acc_3_class:.4f}")
    else:
        buffer.append("\nOverall Accuracy (Classes 1, 2, 3 only): N/A (no samples)")

    # The classification report handles labels automatically
    report_3_class = classification_report(
        y_true_us, y_pred_us,
        labels=class_labels_3,
        target_names=class_names_3,
        zero_division=0
    )
    buffer.append("\nClassification Report (Classes 1, 2, 3):\n")
    buffer.append(report_3_class)
    buffer.append("\n" * 2)

    return "\n".join(buffer)

# ==================
# --- FUNZIONE DI PLOTTING (IL TUO STILE SEABORN) ---
# ==================

def save_seaborn_confusion_matrix(
    cm_abs,  # Accetta la matrice aggregata
    class_names, 
    filename, 
    figsize=FIGSIZE, 
    dpi=DPI,
    label_size=22,
    tick_size=20,
    annot_size=20,
    cbar_tick_size=20
):
    """
    Genera e salva una matrice di confusione in stile Seaborn
    con font personalizzabili, normalizzata per riga (0-100), 
    e SENZA TITOLO.
    """
    try:
        # Calcola percentuali per riga
        row_sums = cm_abs.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1 # Evita divisione per 0
        cm_percent = cm_abs.astype('float') / row_sums * 100
        
        # Crea la figura
        plt.figure(figsize=figsize)
        
        # 1. Crea l'heatmap
        ax = sns.heatmap(
            cm_percent, 
            annot=True, 
            fmt='.2f', 
            cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names,
            annot_kws={'size': annot_size}, 
            vmin=0,                         
            vmax=100                        
        )
        
        # 2. Imposta label (SENZA TITOLO)
        ax.set_xlabel('Predicted', fontsize=label_size) 
        ax.set_ylabel('True', fontsize=label_size)       

        # 3. Imposta tick
        ax.tick_params(axis='x', labelsize=tick_size) 
        ax.tick_params(axis='y', labelsize=tick_size) 
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

        # 4. Modifica colorbar
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=cbar_tick_size) 

        # 5. Salva e chiudi
        plt.tight_layout() 
        plt.savefig(filename, dpi=dpi, bbox_inches='tight')
        plt.close() 
        print(f"    > Plot Seaborn salvato: {filename.name}")

    except Exception as e:
        print(f"    > ERRORE nel plottare con Seaborn: {e}")


# ==================
# --- MAIN SCRIPT (MODIFICATO) ---
# ==================
def main():
    PERCORSO_BASE_DATA = r"C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data"
    base_data_path = Path(PERCORSO_BASE_DATA)
    
    # Directory di output radice per tutte le analisi aggregate
    output_dir_base = base_data_path / "all" / "steady_state" 
    output_dir_base.mkdir(parents=True, exist_ok=True)
    
    print(f"--- Starting Full Aggregation, Metrics, and Plotting ---")
    print(f"Base input directory: {base_data_path}")
    print(f"Base output directory: {output_dir_base}\n")

    # Dictionary per accumulare le matrici di confusione
    session_matrices = {
        session: np.zeros((len(CLASS_LABELS_FULL), len(CLASS_LABELS_FULL)), dtype=int)
        for session in TEST_SESSIONS
    }
    
    soggetti_trovati = sorted(
        [d for d in base_data_path.glob("S*") if d.is_dir()]
    )
    
    if not soggetti_trovati:
        print("ERROR: No subject folders (S*) found.")
        return
        
    print(f"Found {len(soggetti_trovati)} subjects. Starting data collection...")

    # --- 1. Data Collection ---
    soggetti_processati = 0
    for soggetto_path in soggetti_trovati:
        print(f"\nProcessing Subject: {soggetto_path.name}")
        soggetto_processato_con_dati = False
        
        for session_name in TEST_SESSIONS:
            cm_path = soggetto_path / "steady_state" / "evaluation" / session_name / "confusion_matrix_absolute.csv"
            
            if cm_path.exists():
                try:
                    cm_df = pd.read_csv(cm_path, index_col=0)  
                    cm_data_np = cm_df.to_numpy()
                    
                    if cm_data_np.shape == (len(CLASS_LABELS_FULL), len(CLASS_LABELS_FULL)):
                        session_matrices[session_name] += cm_data_np
                        print(f"  > Found and added: {session_name}")
                        soggetto_processato_con_dati = True
                    else:
                        print(f"  > WARNING: {cm_path.name} has unexpected shape {cm_data_np.shape}. Skipping.")
                except Exception as e:
                    print(f"  > ERROR reading {cm_path.name}: {e}")
            else:
                print(f"  > Not found: {session_name}")
        
        if soggetto_processato_con_dati:
            soggetti_processati += 1

    print(f"\n--- Data collection complete. {soggetti_processati} subjects had data. ---")

    # --- 2. Definizione dei Set di Analisi (Individuali e Combinati) ---
    print("\n--- Defining analysis sets and output directories ---")

    # Calcola le matrici combinate
    cm_test1 = session_matrices['test1a'] + session_matrices['test1b']
    cm_test2 = session_matrices['test2a'] + session_matrices['test2b']
    cm_overall = cm_test1 + cm_test2

    # Lista per contenere i set di analisi: (Titolo, Matrice_CM, Dir_Output, Prefisso_File)
    analysis_sets = []
    
    # Aggiungi le sessioni individuali
    for session_name in TEST_SESSIONS:
        out_dir = output_dir_base / session_name
        out_dir.mkdir(parents=True, exist_ok=True)
        analysis_sets.append(
            (f"Session: {session_name}", session_matrices[session_name], out_dir, f"cm_{session_name}")
        )
        print(f"  > Added analysis: {session_name} -> {out_dir.name}")

    # Aggiungi le sessioni combinate
    out_dir_t1 = output_dir_base / "Test1_Combined"
    out_dir_t1.mkdir(parents=True, exist_ok=True)
    analysis_sets.append(
        ("Test 1 (1a+1b)", cm_test1, out_dir_t1, "cm_total_test1")
    )
    print(f"  > Added analysis: Test 1 (1a+1b) -> {out_dir_t1.name}")

    out_dir_t2 = output_dir_base / "Test2_Combined"
    out_dir_t2.mkdir(parents=True, exist_ok=True)
    analysis_sets.append(
        ("Test 2 (2a+2b)", cm_test2, out_dir_t2, "cm_total_test2")
    )
    print(f"  > Added analysis: Test 2 (2a+2b) -> {out_dir_t2.name}")

    out_dir_ovr = output_dir_base / "Overall_Combined"
    out_dir_ovr.mkdir(parents=True, exist_ok=True)
    analysis_sets.append(
        ("Overall (1+2)", cm_overall, out_dir_ovr, "cm_total_overall")
    )
    print(f"  > Added analysis: Overall (1+2) -> {out_dir_ovr.name}")


    # --- 3. & 4. Esecuzione Analisi (CSV, Undersampling, Plot, Metriche) ---
    print("\n--- Starting analysis loop (Unbalanced CSV, Undersampling, Metrics, Plot) ---")
    
    metrics_file_path = output_dir_base / "metrics_report_undersampled.txt"
    all_metrics_str = []
    
    for title, cm_data, output_dir, file_prefix in analysis_sets:
        print(f"\n  Analyzing {title}... (Output dir: {output_dir.name})")
        
        # --- A. Salva CSV Non Bilanciato ---
        save_cm_csv_only(
            cm_data, output_dir, f"{file_prefix}_unbalanced", CLASS_NAMES_FULL
        )

        # --- B. Ricostruisci e Fai Undersampling ---
        y_true, y_pred = reconstruct_predictions(cm_data)
        
        if y_true:
            y_true_us, y_pred_us = perform_undersampling(y_true, y_pred, CLASS_LABELS_FULL)
            
            # --- C. Salva CSV e Plot (solo se ci sono dati post-undersampling) ---
            if y_true_us:
                cm_us = confusion_matrix(y_true_us, y_pred_us, labels=CLASS_LABELS_FULL)
                
                # Salva CSV undersampled
                save_cm_csv_only(
                    cm_us, 
                    output_dir, 
                    f"{file_prefix}_undersampled", 
                    CLASS_NAMES_FULL
                )
                
                # Salva Plot undersampled (CON IL NUOVO STILE SEABORN)
                save_seaborn_confusion_matrix(
                    cm_us, 
                    CLASS_NAMES_FULL,
                    output_dir / f"{file_prefix}_undersampled_seaborn.png"
                )
            
            # --- D. Calcola Metriche (sui dati undersampled) ---
            metrics_str = calculate_and_format_metrics(
                y_true_us, y_pred_us, f"{title} - UNDERSAMPLED",
                CLASS_LABELS_FULL, CLASS_NAMES_FULL,
                CLASS_LABELS_3, CLASS_NAMES_3
            )
            all_metrics_str.append(metrics_str)
        
        else:
            print("    > No data found for this set. Skipping metrics.")
            all_metrics_str.append(f"--- METRICS FOR: {title} ---\nNo data.\n\n")

    # --- 5. Salvataggio Report Metriche Unico ---
    try:
        with open(metrics_file_path, 'w', encoding='utf-8') as f:
            f.write("AGGREGATED METRICS REPORT (UNDERSAMPLED)\n")
            f.write(f"Data Directory: {base_data_path}\n")
            f.write(f"Subjects Analyzed: {soggetti_processati}\n")
            f.write("="*80 + "\n\n")
            f.write("\n".join(all_metrics_str))
        print(f"\n--- Undersampled metrics report saved to: {metrics_file_path} ---")
    except Exception as e:
        print(f"\n--- ERROR saving metrics report: {e} ---")

    print("\n--- Aggregation, metrics, and plotting complete ---")

if __name__ == "__main__":
    main()

--- Starting Full Aggregation, Metrics, and Plotting ---
Base input directory: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data
Base output directory: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\steady_state

Found 12 subjects. Starting data collection...

Processing Subject: S03
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S08
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S10
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S12
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S13
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S15
  > Found and added: te